In [1]:
import tensorflow as tf
import numpy as np
import os
import time

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input
from sklearn.metrics import classification_report


In [2]:
DATASET_PATH = r"D:\Project Jupyter\MODELLING\Dataset_baruPR"
TEST_DIR = os.path.join(DATASET_PATH, "test")

BASE_MODEL_PATH = r"D:\Project Jupyter\FINAL QUANTIZATION FIX\HASIL MODELBASE\mobilenetv3Large_jamur25COPYBARUFIXTF2PREPRO_tf"
TFLITE_INT16_PATH  = r"D:\Project Jupyter\mobilenetv3_BASE_LARGEint16FIXBARU.tflite"

# PARAMETER
IMG_SIZE = (224, 224)
BATCH_SIZE = 1                  # WAJIB 1 untuk INT8



In [3]:
model_fp32 = tf.keras.models.load_model(
    BASE_MODEL_PATH,
    compile=False
)

print("✅ Model FP32 hasil pruning berhasil dimuat")


✅ Model FP32 hasil pruning berhasil dimuat


In [4]:
def preprocess_image(image):
    image = tf.image.resize_with_pad(image, IMG_SIZE[0], IMG_SIZE[1])
    image = preprocess_input(image)
    return image

test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_image
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

class_names = list(test_generator.class_indices.keys())
print("Class:", class_names)


Found 232 images belonging to 2 classes.
Class: ['edible', 'poisonous']


In [11]:
converter = tf.lite.TFLiteConverter.from_saved_model(BASE_MODEL_PATH)

# Aktifkan optimasi
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# Quantize weight ke FP16
converter.target_spec.supported_types = [tf.int16]

print("Converting model ke PTQ INT16")
tflite_fp16_model = converter.convert()

with open(TFLITE_INT16_PATH , "wb") as f:
    f.write(tflite_fp16_model)

model_size_mb = os.path.getsize(TFLITE_INT16_PATH) / (1024 * 1024)
print(f"Model INT16 tersimpan ({model_size_mb:.2f} MB)")

Converting model ke PTQ INT16
Model INT16 tersimpan (11.82 MB)


In [12]:
interpreter = tf.lite.Interpreter(
    model_path=TFLITE_INT16_PATH
)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("Input dtype :", input_details[0]["dtype"])
print("Output dtype:", output_details[0]["dtype"])


Input dtype : <class 'numpy.float32'>
Output dtype: <class 'numpy.float32'>


In [13]:
y_true, y_pred = [], []

start_time = time.time()

for i in range(len(test_generator)):
    image, label = test_generator[i]

    interpreter.set_tensor(
        input_details[0]["index"],
        image.astype(np.float32)
    )
    interpreter.invoke()

    output = interpreter.get_tensor(
        output_details[0]["index"]
    )

    y_true.append(np.argmax(label))
    y_pred.append(np.argmax(output))

end_time = time.time()


In [14]:
total_time = end_time - start_time
avg_time = total_time / len(test_generator)

print("⏱ Inference Time PTQ INT16")
print(f"Total time     : {total_time:.4f} detik")
print(f"Avg per image  : {avg_time:.6f} detik")


⏱ Inference Time PTQ INT16
Total time     : 33.4436 detik
Avg per image  : 0.144153 detik


In [15]:
print("📊 Classification Report (PTQ INT16)")
print(
    classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        digits=4
    )
)


📊 Classification Report (PTQ INT16)
              precision    recall  f1-score   support

      edible     0.9355    0.9667    0.9508       120
   poisonous     0.9630    0.9286    0.9455       112

    accuracy                         0.9483       232
   macro avg     0.9492    0.9476    0.9481       232
weighted avg     0.9487    0.9483    0.9482       232

